In [ ]:
# Project by Mahdi Islam
# Started 5/5/2026
# Goal is to create a machine learning algorithm to learn from listening habits to give better reccomendations

In [ ]:
from tensorflow import keras
from keras import models, layers
import tensorflow as tf
from datetime import datetime
import pandas as pd
import numpy as np
keras.utils.set_random_seed(552026)

In [ ]:
# The following parameters are the inputs for the model
# track_id -- Embedded
# artist_id -- Embedded
# hour_sin
# hour_cos
# day_sin
# day_cos
# target_score


In [ ]:
train = "model_ready_history_train.csv" # input file for bulk training
validation = "model_ready_history_val.csv" # validation file to refine training
test = "model_ready_history_test.csv" # test file to check if its good

# Load the data files
train_df = pd.read_csv(train)
val_df = pd.read_csv(validation)
test_df = pd.read_csv(test)

# Check that the files loaded correctly
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

train_df.head()

In [ ]:
if 'input' in globals():
    del input
    print("Deleted 'input' variable to prevent conflict with built-in function.")

In [ ]:
track_column = "track_id"
artist_column = "artist_id"

numeric_columns = [
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

model_features = [
    track_column,
    artist_column,
] + numeric_columns

target_column = "target_score"

X_train = train_df[model_features]
y_train = train_df[target_column]

X_val = val_df[model_features]
y_val = val_df[target_column]

X_test = test_df[model_features]
y_test = test_df[target_column]

print(X_train.shape, y_train.shape) # check the shape of the data

In [ ]:
# Helper function to aid in embedding for tracks and artists
track_lookup = tf.keras.layers.StringLookup(
    vocabulary=X_train["track_id"].unique(),
    mask_token=None,
    num_oov_indices=1
)

artist_lookup = tf.keras.layers.StringLookup(
    vocabulary=X_train["artist_id"].unique(),
    mask_token=None,
    num_oov_indices=1
)

In [ ]:
track_embedding_dim = 32
artist_embedding_dim = 16

# Track embedding creation
track_embedding = tf.keras.layers.Embedding(
    input_dim=track_lookup.vocabulary_size(),
    output_dim=track_embedding_dim
)
track_input = tf.keras.Input(shape=(1,), name="track_id", dtype=tf.string)
track_index = track_lookup(track_input)
track_vector = track_embedding(track_index)
track_vector = tf.keras.layers.Flatten()(track_vector)

# Artist embedding creation
artist_embedding = tf.keras.layers.Embedding(
    input_dim=artist_lookup.vocabulary_size(),
    output_dim=artist_embedding_dim
)
artist_input = tf.keras.Input(shape=(1,), name="artist_id", dtype=tf.string)
artist_index = artist_lookup(artist_input)
artist_vector = artist_embedding(artist_index)
artist_vector = tf.keras.layers.Flatten()(artist_vector)

In [ ]:
# Creating inputs for the model
hour_sin_input = tf.keras.Input(shape=(1,), name="hour_sin", dtype=tf.float32)
hour_cos_input = tf.keras.Input(shape=(1,), name="hour_cos", dtype=tf.float32)
day_sin_input = tf.keras.Input(shape=(1,), name="day_sin", dtype=tf.float32)
day_cos_input = tf.keras.Input(shape=(1,), name="day_cos", dtype=tf.float32)

In [ ]:
# Concatenating all inputs in one layer
full_input_layer = tf.keras.layers.Concatenate()([
    track_vector,
    artist_vector,
    hour_sin_input,
    hour_cos_input,
    day_sin_input,
    day_cos_input,
])
combined_size = track_embedding_dim + artist_embedding_dim + len(numeric_columns)

In [ ]:
# Creating the model

# Creating the prediction part of the model
prediction_head = models.Sequential([
    # Input layer takes in full_input_layer, with embeddings and numerical inputs
    layers.InputLayer(input_shape=(52,)),
    # First Dense neuron layer with 128 neurons
    layers.Dense(128, activation="relu"),
    # Second Dense neuron layer with 68 neurons
    layers.Dense(64, activation="relu"),
    # Output layer
    layers.Dense(1, activation="sigmoid")
])

# Connecting the input, prediction, and output
output = prediction_head(full_input_layer)

# Defining the model
model = models.Model(
    inputs=[
        track_input,
        artist_input,
        hour_sin_input,
        hour_cos_input,
        day_sin_input,
        day_cos_input
        ],
    outputs=output
)

# Compiling the full model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)
model.summary()

In [ ]:
# Preparing to train the model

# Creating input dictionaries for model training
X_train_dict = {
    "track_id": tf.constant(X_train["track_id"], dtype=tf.string),
    "artist_id": tf.constant(X_train["artist_id"], dtype=tf.string),
    "hour_sin": tf.constant(X_train["hour_sin"], dtype=tf.float32),
    "hour_cos": tf.constant(X_train["hour_cos"], dtype=tf.float32),
    "day_sin": tf.constant(X_train["day_sin"], dtype=tf.float32),
    "day_cos": tf.constant(X_train["day_cos"], dtype=tf.float32),
}

X_val_dict = {
    "track_id": tf.constant(X_val["track_id"], dtype=tf.string),
    "artist_id": tf.constant(X_val["artist_id"], dtype=tf.string),
    "hour_sin": tf.constant(X_val["hour_sin"], dtype=tf.float32),
    "hour_cos": tf.constant(X_val["hour_cos"], dtype=tf.float32),
    "day_sin": tf.constant(X_val["day_sin"], dtype=tf.float32),
    "day_cos": tf.constant(X_val["day_cos"], dtype=tf.float32),
}

X_test_dict = {
    "track_id": tf.constant(X_test["track_id"], dtype=tf.string),
    "artist_id": tf.constant(X_test["artist_id"], dtype=tf.string),
    "hour_sin": tf.constant(X_test["hour_sin"], dtype=tf.float32),
    "hour_cos": tf.constant(X_test["hour_cos"], dtype=tf.float32),
    "day_sin": tf.constant(X_test["day_sin"], dtype=tf.float32),
    "day_cos": tf.constant(X_test["day_cos"], dtype=tf.float32),
}

# Training the model
predictor = model.fit(
    X_train_dict,
    y_train,
    validation_data=(X_val_dict, y_val),
    epochs=20,
    batch_size=32
)

In [ ]:
test_loss, test_mae = model.evaluate(X_test_dict, y_test)

print("Test loss:", test_loss)
print("Test MAE:", test_mae)

In [ ]:
# Generate predictions on the test set
predictions = model.predict(X_test_dict, verbose=0).flatten()

# Create a results table from the test input data
test_results = X_test.copy().reset_index(drop=True)

# Add predicted score, actual score, and error
test_results["predicted_score"] = predictions
test_results["actual_score"] = y_test.reset_index(drop=True)
test_results["error"] = abs(test_results["actual_score"] - test_results["predicted_score"])

# Put index into its own column
clean_predictions = test_results.reset_index()

# Rename columns for readability
clean_predictions = clean_predictions.rename(columns={
    "index": "row",
    "artist_id": "artist",
    "predicted_score": "predicted",
    "actual_score": "actual"
})

# Round numerical columns
round_columns = [
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "predicted",
    "actual",
    "error"
]

clean_predictions[round_columns] = clean_predictions[round_columns].round(3)

# Reorder columns
clean_predictions = clean_predictions[
    [
        "row",
        "track_id",
        "artist",
        "hour_sin",
        "hour_cos",
        "day_sin",
        "day_cos",
        "predicted",
        "actual",
        "error",
    ]
]

# Display table
clean_predictions.head(25)

In [ ]:
# Helper function to convert hour into sine/cosine features
def encode_hour(hour):
    hour = int(hour)
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    return hour_sin, hour_cos

# Helper function to convert day of week into sine/cosine features
# Monday = 0, Tuesday = 1, ..., Sunday = 6
def encode_day(day_of_week):
    day_of_week = int(day_of_week)
    day_sin = np.sin(2 * np.pi * day_of_week / 7)
    day_cos = np.cos(2 * np.pi * day_of_week / 7)
    return day_sin, day_cos

# Ask user for input
track_id = input("Enter track_id, example spotify:track:... : ")
artist_id = input("Enter artist_id / artist name: ")

hour = input("Enter hour of day, 0-23: ")
day_of_week = input("Enter day of week, Monday=0, Tuesday=1, ..., Sunday=6: ")

# Encode time features
hour_sin, hour_cos = encode_hour(hour)
day_sin, day_cos = encode_day(day_of_week)

# Create model input dictionary, ensuring correct TensorFlow dtypes
single_input = {
    "track_id": tf.constant([track_id], dtype=tf.string),
    "artist_id": tf.constant([artist_id], dtype=tf.string),
    "hour_sin": tf.constant([hour_sin], dtype=tf.float32),
    "hour_cos": tf.constant([hour_cos], dtype=tf.float32),
    "day_sin": tf.constant([day_sin], dtype=tf.float32),
    "day_cos": tf.constant([day_cos], dtype=tf.float32),
}

# Predict score
predicted_score = model.predict(single_input, verbose=0)[0][0]

print("Predicted target score:", round(float(predicted_score), 3))